# ☀️ Solar Filament Segmentation 2026: Competition Pipeline (U-Net++ & 5-Fold Ensemble)

This notebook implements a high-performance segmentation pipeline:
- **COCO Polygon Parsing**: Accurate mask rasterization fixing the empty-mask baseline bug.
- **5-Fold Stratified Cross-Validation**: Balanced folds based on filament area coverage.
- **Architecture**: U-Net++ with `efficientnet-b4` encoder and scSE attention.
- **Loss Function**: Hybrid BCE (0.3) + Dice (0.4) + Focal Loss (0.3).
- **Augmentation Pipeline**: CLAHE, RandomRotate90, ShiftScaleRotate, Brightness/Contrast.
- **Optimization**: AdamW + CosineAnnealingWarmRestarts + PyTorch AMP.
- **Inference**: 5-Fold Model Ensembling + 4x Test-Time Augmentation (TTA) + OOF Optimal Thresholding.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations pycocotools timm

In [ ]:
import os
import sys
import json
import time
import glob
import math
import warnings
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import pycocotools.mask as mask_utils

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def get_default_paths():
    kaggle_paths = [
        Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
        Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
    ]
    for p in kaggle_paths:
        if p.exists():
            return p, Path("/kaggle/working")
    return Path("./"), Path("./")

DATA_DIR, OUTPUT_DIR = get_default_paths()

CFG = {
    'seed': 42,
    'img_size': (512, 512),
    'batch_size': 8,
    'epochs': 25,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'folds': 5,
    'arch': 'UnetPlusPlus',
    'backbone': 'efficientnet-b4',
    'encoder_weights': 'imagenet',
    'num_workers': 2 if torch.cuda.is_available() else 0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'loss_weights': {'bce': 0.3, 'dice': 0.4, 'focal': 0.3},
    'data_dir': DATA_DIR,
    'output_dir': OUTPUT_DIR
}

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
print(f"Device: {CFG['device']} | Data: {CFG['data_dir']}")

In [ ]:
def parse_coco_annotations(ann_path):
    with open(ann_path, 'r') as f:
        coco_data = json.load(f)
    img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
    filename_to_anns = defaultdict(list)
    for ann in coco_data["annotations"]:
        img_fn = img_id_to_filename.get(ann["image_id"])
        if img_fn:
            filename_to_anns[img_fn].append(ann.get("segmentation", []))
    return filename_to_anns

def create_mask_from_segmentations(seg_list, img_shape):
    h, w = img_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    if not seg_list:
        return mask
    for seg_entry in seg_list:
        if not seg_entry:
            continue
        if isinstance(seg_entry, list):
            for poly in seg_entry:
                if isinstance(poly, list) and len(poly) >= 6:
                    pts = np.array(poly, dtype=np.float32).reshape((-1, 2)).astype(np.int32)
                    cv2.fillPoly(mask, [pts], 1)
                elif isinstance(poly, np.ndarray) and poly.size >= 6:
                    pts = poly.reshape((-1, 2)).astype(np.int32)
                    cv2.fillPoly(mask, [pts], 1)
    return mask

In [ ]:
class SolarDataset(Dataset):
    def __init__(self, df, img_dir, annotations=None, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.annotations = annotations
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx]['filename']
        img_path = os.path.join(self.img_dir, filename)
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, filename

        ann_entry = self.annotations.get(filename, []) if self.annotations else []
        mask = create_mask_from_segmentations(ann_entry, (h, w))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        mask = mask.unsqueeze(0).float()
        return image, mask

def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size[0], img_size[1]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=30, p=0.5, border_mode=cv2.BORDER_CONSTANT),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size[0], img_size[1]),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

In [ ]:
class HybridLoss(nn.Module):
    def __init__(self, weights={'bce': 0.3, 'dice': 0.4, 'focal': 0.3}):
        super(HybridLoss, self).__init__()
        self.weights = weights
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.25, gamma=2.0)

    def forward(self, y_pred, y_true):
        loss_bce = self.bce(y_pred, y_true)
        loss_dice = self.dice(y_pred, y_true)
        loss_focal = self.focal(y_pred, y_true)
        return (self.weights['bce'] * loss_bce +
                self.weights['dice'] * loss_dice +
                self.weights['focal'] * loss_focal)

def build_model(arch=CFG['arch'], backbone=CFG['backbone']):
    model_func = getattr(smp, arch, smp.UnetPlusPlus)
    try:
        model = model_func(
            encoder_name=backbone,
            encoder_weights=CFG['encoder_weights'],
            in_channels=3,
            classes=1,
            decoder_attention_type='scse',
            activation=None
        )
    except Exception:
        model = model_func(
            encoder_name=backbone,
            encoder_weights=CFG['encoder_weights'],
            in_channels=3,
            classes=1,
            activation=None
        )
    return model

def calculate_dice(y_pred, y_true, threshold=0.5, smooth=1e-6):
    y_pred = (y_pred > threshold).float()
    intersection = (y_pred * y_true).sum(dim=(2, 3))
    total = y_pred.sum(dim=(2, 3)) + y_true.sum(dim=(2, 3))
    dice = (2. * intersection + smooth) / (total + smooth)
    return dice.mean().item()

def mask_to_coco_rle(binary_mask):
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    rle['counts'] = rle['counts'].decode('utf-8')
    return rle['counts']

def predict_tta(model, images):
    logits_orig = model(images)
    probs_orig = torch.sigmoid(logits_orig)

    logits_h = model(torch.flip(images, dims=[3]))
    probs_h = torch.flip(torch.sigmoid(logits_h), dims=[3])

    logits_v = model(torch.flip(images, dims=[2]))
    probs_v = torch.flip(torch.sigmoid(logits_v), dims=[2])

    logits_hv = model(torch.flip(images, dims=[2, 3]))
    probs_hv = torch.flip(torch.sigmoid(logits_hv), dims=[2, 3])

    return (probs_orig + probs_h + probs_v + probs_hv) / 4.0

In [ ]:
# Execute 5-Fold Training & Generate submission.csv
from solaris_pipeline import run_pipeline
run_pipeline()